<a href="https://colab.research.google.com/github/UDHAYA046/sem5_ML/blob/main/ML_udh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Upload your zip


In [1]:
from google.colab import files
up = files.upload()   # choose Labelled_excels.zip from your Downloads


Saving Labelled_excels.zip to Labelled_excels.zip


Unzip and see contents


In [2]:
!mkdir -p /content/excels
!unzip -o "Labelled_excels.zip" -d /content/excels
!ls -lah /content/excels | head -n 30


Archive:  Labelled_excels.zip
  inflating: /content/excels/output_s - 149.xlsx  
  inflating: /content/excels/output_s - 153.xlsx  
  inflating: /content/excels/output_s - 154.xlsx  
  inflating: /content/excels/output_s - 155.xlsx  
  inflating: /content/excels/output_s - 156.xlsx  
  inflating: /content/excels/output_s - 157.xlsx  
  inflating: /content/excels/output_s - 158.xlsx  
  inflating: /content/excels/output_s - 160.xlsx  
  inflating: /content/excels/output_s - 161.xlsx  
  inflating: /content/excels/output_s - 162.xlsx  
  inflating: /content/excels/output_s - 163.xlsx  
  inflating: /content/excels/output_s - 165.xlsx  
  inflating: /content/excels/output_s - 166.xlsx  
  inflating: /content/excels/output_s - 167.xlsx  
  inflating: /content/excels/output_s - 134.xlsx  
  inflating: /content/excels/output_s - 135.xlsx  
  inflating: /content/excels/output_s - 136.xlsx  
  inflating: /content/excels/output_s - 137.xlsx  
  inflating: /content/excels/output_s - 138.xlsx  
 

Read all Excels → one dataframe

In [3]:
import os, re, pandas as pd
from glob import glob

EXCEL_DIR = "/content/excels"   # <- where you unzipped
MANIFEST_DIR = "/content/manifests"
os.makedirs(MANIFEST_DIR, exist_ok=True)

def get_student_id(fname):
    # e.g., "output_s - 134.xlsx" → "134"
    m = re.search(r'(\d+)', os.path.basename(fname))
    return m.group(1) if m else None

all_rows = []
excel_paths = sorted(glob(os.path.join(EXCEL_DIR, "*.xlsx")))

print(f"Found {len(excel_paths)} excel files")

for path in excel_paths:
    sid = get_student_id(path)
    df = pd.read_excel(path)
    # standardize column names just in case
    df = df.rename(columns={
        "Start time":"start_s",
        "End time":"end_s",
        "File Name":"clip_name",
        "Label":"label",
        "Member":"member",
        "Number":"number"
    })
    # essential columns exist?
    missing = {"start_s","end_s","clip_name","label"} - set(df.columns)
    if missing:
        print(f"⚠️ {os.path.basename(path)} missing columns: {missing}")
        continue

    df["student_id"] = sid
    df["duration_s"] = df["end_s"] - df["start_s"]
    # keep only the columns we care about
    keep = ["student_id","number","clip_name","start_s","end_s","duration_s","label","member"]
    all_rows.append(df[keep])

manifest = pd.concat(all_rows, ignore_index=True)
print("Shape:", manifest.shape)
manifest.head()


Found 28 excel files
Shape: (934, 8)


,student_id,number,clip_name,start_s,end_s,duration_s,label,member
0,134,1,134s1,6.087315,8.507075,2.419760,3.0,S
1,134,2,134s2,15.014376,18.531601,3.517225,2.0,S
2,134,3,134s3,26.835345,31.410509,4.575164,2.0,S
3,134,4,134s4,35.728456,37.859864,2.131408,3.0,S
4,134,5,134s5,42.544762,55.724880,13.180119,1.0,S


Basic sanity checks

In [4]:
# label range and missing
print("Label counts:\n", manifest["label"].value_counts().sort_index())
print("\nAny NaNs?", manifest.isna().sum())

# duration sanity
print("\nDuration stats (sec):")
print(manifest["duration_s"].describe())

# duplicates of clip_name across different students?
dups = manifest["clip_name"].duplicated().sum()
print(f"\nDuplicate clip_name entries: {dups}")


Label counts:
 label
1.0    140
2.0    191
3.0    299
4.0    197
5.0    106
Name: count, dtype: int64

Any NaNs? student_id    0
number        0
clip_name     0
start_s       0
end_s         0
duration_s    0
label         1
member        0
dtype: int64

Duration stats (sec):
count    934.000000
mean       8.466790
std       11.656017
min        0.283362
25%        1.736890
50%        4.005822
75%       10.209933
max      127.448423
Name: duration_s, dtype: float64

Duplicate clip_name entries: 0


3) Save the raw manifest

In [5]:
RAW_MANIFEST = os.path.join(MANIFEST_DIR, "dataset_manifest_raw.csv")
manifest.to_csv(RAW_MANIFEST, index=False)
print("✅ Saved:", RAW_MANIFEST)


✅ Saved: /content/manifests/dataset_manifest_raw.csv


Make a “clean” manifest (very gentle filtering only)

In [6]:
clean = manifest.copy()

# keep only sensible labels and durations > 0
clean = clean[
    clean["label"].between(1,5, inclusive="both") &
    (clean["duration_s"] > 0)
].reset_index(drop=True)

# (Optional) drop ultra-short/ultra-long outliers; tweak later if needed
clean = clean.query("duration_s >= 0.7 and duration_s <= 15").reset_index(drop=True)

CLEAN_MANIFEST = os.path.join(MANIFEST_DIR, "dataset_manifest_clean.csv")
clean.to_csv(CLEAN_MANIFEST, index=False)

print("✅ Saved clean manifest:", CLEAN_MANIFEST)
print("Counts by label:\n", clean["label"].value_counts().sort_index())
print("\nPer-student rows (first 10):")
print(clean.groupby("student_id").size().sort_index().head(10))
clean.head()


✅ Saved clean manifest: /content/manifests/dataset_manifest_clean.csv
Counts by label:
 label
1.0    114
2.0    165
3.0    216
4.0    144
5.0     96
Name: count, dtype: int64

Per-student rows (first 10):
student_id
134    29
135    22
136    27
137    44
138    20
139    20
140    22
141    43
142    34
143    21
dtype: int64


,student_id,number,clip_name,start_s,end_s,duration_s,label,member
0,134,1,134s1,6.087315,8.507075,2.419760,3.0,S
1,134,2,134s2,15.014376,18.531601,3.517225,2.0,S
2,134,3,134s3,26.835345,31.410509,4.575164,2.0,S
3,134,4,134s4,35.728456,37.859864,2.131408,3.0,S
4,134,5,134s5,42.544762,55.724880,13.180119,1.0,S


Quick summary table

In [7]:
summary = pd.DataFrame({
    "clips_total":[len(clean)],
    "students_total":[clean['student_id'].nunique()],
    "avg_duration_s":[round(clean['duration_s'].mean(),3)],
})
by_label = clean["label"].value_counts().sort_index()
print("Summary:\n", summary.to_string(index=False))
print("\nLabel distribution:\n", by_label.to_string())


Summary:
  clips_total  students_total  avg_duration_s
         735              28           4.421

Label distribution:
 label
1.0    114
2.0    165
3.0    216
4.0    144
5.0     96


Upload the cleaned audio files

In [8]:
from google.colab import files
up = files.upload()   # choose cleaned_audio.zip


Saving Cleaned_Audios.zip to Cleaned_Audios.zip


In [10]:
!unzip -o "Cleaned_Audios.zip" -d /content/cleaned_audio
!ls -lah /content/cleaned_audio | head -n 20


Archive:  Cleaned_Audios.zip
   creating: /content/cleaned_audio/Cleaned_Audios/
  inflating: /content/cleaned_audio/Cleaned_Audios/101_s1.mp3  
  inflating: /content/cleaned_audio/Cleaned_Audios/101_s10.mp3  
  inflating: /content/cleaned_audio/Cleaned_Audios/101_s11.mp3  
  inflating: /content/cleaned_audio/Cleaned_Audios/101_s12.mp3  
  inflating: /content/cleaned_audio/Cleaned_Audios/101_s13.mp3  
  inflating: /content/cleaned_audio/Cleaned_Audios/101_s14.mp3  
  inflating: /content/cleaned_audio/Cleaned_Audios/101_s15.mp3  
  inflating: /content/cleaned_audio/Cleaned_Audios/101_s16.mp3  
  inflating: /content/cleaned_audio/Cleaned_Audios/101_s17.mp3  
  inflating: /content/cleaned_audio/Cleaned_Audios/101_s18.mp3  
  inflating: /content/cleaned_audio/Cleaned_Audios/101_s19.mp3  
  inflating: /content/cleaned_audio/Cleaned_Audios/101_s2.mp3  
  inflating: /content/cleaned_audio/Cleaned_Audios/101_s20.mp3  
  inflating: /content/cleaned_audio/Cleaned_Audios/101_s21.mp3  
  inflating

Merge Excels with actual audio file locations

what audio files we have


In [12]:
import os, re
from glob import glob

AUDIO_DIR = "/content/cleaned_audio/Cleaned_Audios"

audio_paths = glob(os.path.join(AUDIO_DIR, "*.mp3"))
audio_basenames = [os.path.splitext(os.path.basename(p))[0] for p in audio_paths]

# show a few
print("Total MP3s:", len(audio_basenames))
print(sorted(audio_basenames)[:20])


Total MP3s: 3904
['101_s1', '101_s10', '101_s11', '101_s12', '101_s13', '101_s14', '101_s15', '101_s16', '101_s17', '101_s18', '101_s19', '101_s2', '101_s20', '101_s21', '101_s22', '101_s23', '101_s24', '101_s25', '101_s26', '101_s27']


Which student IDs exist in audio vs Excel?

In [13]:
import pandas as pd, re, os
from glob import glob

EXCEL_DIR = "/content/excels"
excel_sid = set()
for f in glob(os.path.join(EXCEL_DIR, "*.xlsx")):
    m = re.search(r'(\d+)', os.path.basename(f))
    if m: excel_sid.add(m.group(1))

audio_sid = set()
for b in audio_basenames:
    m = re.match(r'(\d+)', b)          # e.g., "286_s3" -> "286"
    if m: audio_sid.add(m.group(1))

print("Students in Excel:", len(excel_sid), sorted(list(excel_sid))[:10], "…")
print("Students in Audio:", len(audio_sid), sorted(list(audio_sid))[:10], "…")

print("\nMissing audio for these Excel students:")
print(sorted(list(excel_sid - audio_sid)))


Students in Excel: 28 ['134', '135', '136', '137', '138', '139', '140', '141', '142', '143'] …
Students in Audio: 142 ['101', '102', '103', '104', '105', '106', '107', '108', '110', '111'] …

Missing audio for these Excel students:
['134', '135']


Identify which Excel rows have missing labels

In [15]:
import os, re, pandas as pd
from glob import glob

EXCEL_DIR = "/content/excels"

bad = []
for path in sorted(glob(os.path.join(EXCEL_DIR, "*.xlsx"))):
    sid = re.search(r'(\d+)', os.path.basename(path)).group(1)
    df = pd.read_excel(path)
    if "Label" not in df.columns or "File Name" not in df.columns:
        print("⚠️ Missing columns in:", os.path.basename(path));
        continue
    df["_label"] = pd.to_numeric(df["Label"], errors="coerce")  # NaN if empty
    miss_rows = df[df["_label"].isna()]
    if len(miss_rows):
        bad.append((sid, os.path.basename(path), len(miss_rows)))
        # print a few examples
        print(f"⚠️ {os.path.basename(path)} has {len(miss_rows)} rows with NaN label. Examples:")
        display(miss_rows.head(3)[["File Name","Label"]])

if not bad:
    print("✅ No NaN labels found")


⚠️ output_s - 148.xlsx has 1 rows with NaN label. Examples:


,File Name,Label
51,148s52,NaN


Rebuild the matcher (skip NaNs + normalize clip names)

In [16]:
import os, re, pandas as pd
from glob import glob

AUDIO_DIR = "/content/cleaned_audio/Cleaned_Audios"

# Collect available audio basenames (lowercased)
audio_paths = glob(os.path.join(AUDIO_DIR, "*.mp3"))
audio_set = set(os.path.splitext(os.path.basename(p))[0].lower() for p in audio_paths)
audio_map = {os.path.splitext(os.path.basename(p))[0].lower(): p for p in audio_paths}

def variants(clip):
    """Generate possible name variants:
       '134s1' -> ['134s1','134_s1','134_s01','134s01'] (all lower, no spaces)."""
    base = str(clip).strip().replace(" ", "").lower()
    a = base
    b = base if "_s" in base else base.replace("s", "_s", 1)
    # zero-padded
    c = re.sub(r"_s(\d+)$", lambda m: f"_s{int(m.group(1)):02d}", b)
    d = re.sub(r"s(\d+)$",  lambda m: f"_s{int(m.group(1)):02d}", a)
    return list(dict.fromkeys([a, b, c, d]))

rows = []   # (sid, clip_from_excel, chosen_audio_key, label_int)
miss = []   # track missing for debugging

for path in sorted(glob(os.path.join(EXCEL_DIR, "*.xlsx"))):
    sid = re.search(r'(\d+)', os.path.basename(path)).group(1)
    df = pd.read_excel(path).rename(columns={"File Name":"FileName"})
    if "FileName" not in df.columns or "Label" not in df.columns:
        print("⚠️ Missing columns in:", os.path.basename(path))
        continue
    # safe numeric labels; drop NaNs
    df["_label"] = pd.to_numeric(df["Label"], errors="coerce")
    df = df.dropna(subset=["_label"])
    # round to nearest int and clip to [1,5]
    df["_label"] = df["_label"].round().clip(1,5).astype(int)

    for _, r in df.iterrows():
        clip = r["FileName"]
        label = int(r["_label"])
        chosen = None
        for v in variants(clip):
            if v in audio_set:
                chosen = v
                break
        if chosen:
            rows.append((sid, clip, chosen, label))
        else:
            miss.append((sid, str(clip)))

print(f"Found matches for {len(rows)} clips; Missing {len(miss)}")
print("Sample misses (first 15):", miss[:15])


Found matches for 477 clips; Missing 456
Sample misses (first 15): [('134', '134s1'), ('134', '134s2'), ('134', '134s3'), ('134', '134s4'), ('134', '134s5'), ('134', '134s6'), ('134', '134s7'), ('134', '134s8'), ('134', '134s9'), ('134', '134s10'), ('134', '134s11'), ('134', '134s12'), ('134', '134s13'), ('134', '134s14'), ('134', '134s15')]


Save the final manifest using the best matches

In [17]:
import pandas as pd, os

OUT = "/content/dataset_manifest_final.csv"

records = []
for sid, clip, chosen_key, label in rows:
    records.append({
        "student_id": sid,
        "clip_name": chosen_key,                     # normalized name that exists
        "file_path": audio_map[chosen_key],          # absolute path in Colab
        "label": int(label)
    })

manifest = pd.DataFrame(records).drop_duplicates(subset=["file_path"])
manifest.to_csv(OUT, index=False)

print("✅ Saved:", OUT, "| clips:", len(manifest), "| students:", manifest.student_id.nunique())
print("Label distribution:\n", manifest.label.value_counts().sort_index())
manifest.head()


✅ Saved: /content/dataset_manifest_final.csv | clips: 477 | students: 13
Label distribution:
 label
1     65
2     69
3    124
4    123
5     96
Name: count, dtype: int64


,student_id,clip_name,file_path,label
0,136,136_s2,/content/cleaned_audio/Cleaned_Audios/136_s2.mp3,5
1,136,136_s3,/content/cleaned_audio/Cleaned_Audios/136_s3.mp3,2
2,136,136_s4,/content/cleaned_audio/Cleaned_Audios/136_s4.mp3,3
3,136,136_s5,/content/cleaned_audio/Cleaned_Audios/136_s5.mp3,3
4,136,136_s6,/content/cleaned_audio/Cleaned_Audios/136_s6.mp3,3


quickly see which students are entirely missing

In [18]:
# Students present in Excels vs in audio
excel_sid = sorted({re.search(r'(\d+)', os.path.basename(p)).group(1)
                    for p in glob(os.path.join(EXCEL_DIR, "*.xlsx"))})
audio_sid  = sorted({re.match(r'(\d+)', k).group(1) for k in audio_set})

missing_students = sorted(set(excel_sid) - set(audio_sid))
print("Students with Excel but no audio uploaded:", missing_students)


Students with Excel but no audio uploaded: ['134', '135']


NOW , proceeding with this , will scale it to more students once audio files are available

---


Clips linked: 477 clips from 13 students

---


Labels OK: {1:65, 2:69, 3:124, 4:123, 5:96}

---


One Excel row with NaN label: output_s - 148.xlsx → 148s52 (we safely skipped it)

---


Students missing in audio ZIP: ['134', '135'] (present in Excels, but their MP3s weren’t in Cleaned_Audios.zip)

---


Name pattern handled: ID_sN.mp3 (e.g., 136_s2.mp3)

************************************************************

Save everything to Drive (so your team can access)


In [21]:
from google.colab import files
files.download("/content/dataset_manifest_final.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>